In [ ]:
import os
import json
import time
import random

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from datetime import datetime, timezone
from pathlib import Path

import requests
import pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential_jitter, retry_if_exception_type

BASE_URL = "https://api.github.com"
API_VERSION = "2022-11-28"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
TARGET_REPO = "pandas-dev/pandas"
STATE_FILE = Path("state.json")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": API_VERSION,
    "User-Agent": "lab3"
})

session.timeout = 305

In [ ]:
def safe_get_json(response, url):
    # Проверка Content-Type
    content_type = response.headers.get("Content-Type", "")
    if "application/json" not in content_type:
        raise ValueError(
            f"Ожидался JSON, получен '{content_type}'. "
            f"Первые 200 байт: {response.text[:200]}"
        )
    # Проверка статуса
    if response.status_code != 200:
        raise requests.HTTPError(
            f"HTTP {response.status_code} для {url}. "
            f"Ответ: {response.text[:200]}",
            response=response
        )
    return response.json()

In [3]:
def _request_with_retry(session, url, params = None, max_retries = 5):
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, params=params, timeout=30)
        except requests.exceptions.Timeout:
            if attempt == max_retries:
                raise
            delay = min(60, (2 ** attempt) + random.uniform(0, 1))
            logging.warning(f"Timeout, попытка {attempt}/{max_retries}, пауза {delay:.1f}с")
            time.sleep(delay)
            continue

        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 0))
            delay = max(retry_after, min(60, (2 ** attempt) + random.uniform(0, 1)))
            logging.warning(f"429, попытка {attempt}/{max_retries}, пауза {delay:.1f}с")
            time.sleep(delay)
            continue

        if 500 <= response.status_code < 600:
            if attempt == max_retries:
                response.raise_for_status()
            delay = min(60, (2 ** attempt) + random.uniform(0, 1))
            logging.warning(f"{response.status_code}, попытка {attempt}/{max_retries}, пауза {delay:.1f}с")
            time.sleep(delay)
            continue

        if 400 <= response.status_code < 500:
            response.raise_for_status()

        return response

    raise Exception(f"Исчерпаны попытки для {url}")

In [4]:
def parse_link_header(link_header):
    links = {}
    if not link_header:
        return links
    for part in link_header.split(","):
        part = part.strip()
        if not part:
            continue
        url_part, rel_part = part.split(";", 1)
        url = url_part.strip().strip("<>")
        rel = rel_part.strip().replace("rel=", "").strip('"')
        links[rel] = url
    return links


def collect_issues_paginated(repo, page_size: int = 100, pages: int = 20):
    records = []
    log = []
    url = f"{BASE_URL}/repos/{repo}/issues"
    params = {
        "state": "all",
        "per_page": page_size,
        "sort": "created",
        "direction": "asc"
    }

    page_num = 0
    while url and page_num < pages:
        page_num += 1
        start = time.monotonic()
        try:
            resp = _request_with_retry(session, url, params)
            data = safe_get_json(resp, url)
            duration = time.monotonic() - start

            if not isinstance(data, list):
                logging.warning(f"Страница {page_num}: неожиданный формат ответа")
                break

            for item in data:
                item["_collected_at"] = datetime.now(timezone.utc).isoformat()

            records.extend(data)

            log.append({
                "page": page_num,
                "url": url,
                "params": json.dumps(params),
                "status_code": resp.status_code,
                "records_count": len(data),
                "duration_sec": round(duration, 3),
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "rate_limit_remaining": resp.headers.get("X-RateLimit-Remaining"),
                "rate_limit_reset": resp.headers.get("X-RateLimit-Reset")
            })

            links = parse_link_header(resp.headers.get("Link", ""))
            url = links.get("next")
            params = {}

            if len(data) == 0:
                logging.info(f"Страница {page_num } пустая — остановка")
                break
            
            logging.info(f"Собрано {len(data)} записей на странице №{page_num}")
            
            time.sleep(1.0)
        
        except Exception as e:
            logging.error(f"Страница {page_num}: ошибка — {e}")
            log.append({
                "page": page_num,
                "url": url,
                "status_code": getattr(getattr(e, "response", None), "status_code", None),
                "records_count": 0,
                "duration_sec": round(time.monotonic() - start, 3),
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "error": str(e)[:200]
            })
            break
    logging.info(f"Всего собрано {len(records)} записей за {page_num} страниц")
    return records, log

In [5]:
def normalize_issues(records):
    if not records:
        return pd.DataFrame()

    df_labels = pd.json_normalize(
        records,
        record_path="labels",
        meta=[
            "id", "number", "title", "state", "created_at", "closed_at",
            "updated_at", "comments", "user.login", "user.id",
            "user.type", "author_association", "html_url", "body",
            "_collected_at"
        ],
        meta_prefix="issue_",
        sep=".",
        errors="ignore"
    )
    if df_labels.empty:
        df_labels = pd.json_normalize(records, sep=".", errors="ignore")
    else:
        df_labels = df_labels.groupby("issue_id", as_index=False).agg({
            "name": lambda x: "|".join(str(v) for v in x if pd.notna(v)),
            **{c: "first" for c in df_labels.columns if c.startswith("issue_") and c != "issue_id"}
        })
        df_labels = df_labels.rename(columns={"name": "labels"})
    df_assignees = pd.json_normalize(
        records,
        record_path="assignees",
        meta=["id", "number"],
        meta_prefix="issue_",
        sep=".",
        errors="ignore"
    )
    if not df_assignees.empty:
        df_assignees = df_assignees.groupby("issue_id", as_index=False).agg({
            "login": lambda x: "|".join(str(v) for v in x if pd.notna(v))
        }).rename(columns={"login": "assignees_logins"})
        df_labels = df_labels.merge(df_assignees, on="issue_id", how="left")

    return df_labels

In [6]:
def enrich_with_repo(df, repo):
    repo_url = f"{BASE_URL}/repos/{repo}"
    repo_resp = _request_with_retry(session, repo_url)
    repo_data = safe_get_json(repo_resp, repo_url)

    repo_info = {
        "repo_full_name": repo_data.get("full_name"),
        "repo_language": repo_data.get("language"),
        "repo_stars": repo_data.get("stargazers_count"),
        "repo_forks": repo_data.get("forks_count"),
        "repo_open_issues": repo_data.get("open_issues_count"),
        "repo_created_at": repo_data.get("created_at"),
    }
    
    for k, v in repo_info.items():
        df[k] = v

    logging.info(f"Строк до обогащения: {len(df)}, после: {len(df)}")
    assert len(df) > 0, "Таблица пуста после обогащения"
    return df

In [7]:
def check_data_quality(df):
    report = {}
    cols_for_quality = [x for x in df.columns if 'user' not in x]
    completeness = (df[cols_for_quality].notna().mean() * 100).round(2).to_dict()
    report["completeness"] = completeness

    dup_count = df.duplicated(subset=["issue_id"]).sum()
    report["duplicates_by_key"] = int(dup_count)
    
    for col in ["issue_created_at", "issue_closed_at", "issue_updated_at"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    closed_mask = df["issue_closed_at"].notna() & df["issue_created_at"].notna()
    valid_lifetime = (df.loc[closed_mask, "issue_closed_at"] >=
                      df.loc[closed_mask, "issue_created_at"]).all()
    
    report["business_rule_closed_after_created"] = bool(valid_lifetime)
    report["business_rule_comments_non_negative"] = bool((df["issue_comments"] >= 0).all())

    df["lifetime_days"] = (
        df["issue_closed_at"] - df["issue_created_at"]
    ).dt.total_seconds() / 86400.0

    return report

In [8]:
def load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    return {}

def save_state(state):
    STATE_FILE.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

def incremental_filter(df, state):
    watermark = state.get("issues_watermark")
    if watermark is None:
        return df
    
    watermark_dt = pd.to_datetime(watermark, utc=True)
    df["issue_created_at"] = pd.to_datetime(df["issue_created_at"], utc=True, errors="coerce")
    new_df = df[df["issue_created_at"] > watermark_dt].copy()
    logging.info(f"Инкрементально новых записей: {len(new_df)} (из {len(df)})")
    return new_df

def update_watermark(df, state):
    if df.empty:
        return state
    max_created = pd.to_datetime(df["issue_created_at"], utc=True).max()
    state["issues_watermark"] = max_created.isoformat()
    state["last_run"] = datetime.now(timezone.utc).isoformat()
    return state

In [9]:
def compute_metrics(df):
    df["created_month"] = pd.to_datetime(df["issue_created_at"], utc=True).dt.to_period("M")
    df["closed_month"] = pd.to_datetime(df["issue_closed_at"], utc=True).dt.to_period("M")

    opened = df.groupby("created_month").size().rename("opened")
    closed = df.groupby("closed_month").size().rename("closed")

    metrics = pd.concat([opened, closed], axis=1).fillna(0).astype(int)
    metrics["net"] = metrics["opened"] - metrics["closed"]

    lifetime = df.groupby("created_month")["lifetime_days"].mean().round(2)

    return metrics.join(lifetime.rename("avg_lifetime_days"))

## Запуск функций

In [10]:
# 1. Сбор
records, log_records = collect_issues_paginated(TARGET_REPO, pages=5)

# 2. Сохранение сырых данных
with open("data_raw.json", "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# 3. Журнал сбора
log_df = pd.DataFrame(log_records)
log_df.to_csv("collection_log.csv", index=False, encoding="utf-8")

2026-10-06 20:50:35,781 INFO Собрано 100 записей на странице №1
2026-10-06 20:50:37,657 INFO Собрано 100 записей на странице №2
2026-10-06 20:50:39,598 INFO Собрано 100 записей на странице №3
2026-10-06 20:50:41,366 INFO Собрано 100 записей на странице №4
2026-10-06 20:50:43,270 INFO Собрано 100 записей на странице №5
2026-10-06 20:50:44,276 INFO Всего собрано 500 записей за 5 страниц


In [11]:
# 4. Нормализация
df = normalize_issues(records)

# 5. Обогащение
df = enrich_with_repo(df, TARGET_REPO)
df

2026-10-06 20:50:47,339 INFO Строк до обогащения: 362, после: 362


,issue_id,labels,issue_number,issue_title,issue_state,issue_created_at,issue_closed_at,issue_updated_at,issue_comments,issue_user.login,...,issue_html_url,issue_body,issue__collected_at,assignees_logins,repo_full_name,repo_language,repo_stars,repo_forks,repo_open_issues,repo_created_at
0,339355,Bug,8,np.fix doesn't work,closed,2010-09-29T19:45:47Z,2011-06-23T04:50:05Z,2014-02-11T20:40:01Z,3,None,...,https://github.com/pandas-dev/pandas/issues/8,"```\nIn [28]: np.fix(Series([1,2,3], range(3))...",2026-10-06T15:50:35.781439+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
1,352369,Enhancement,13,Add PyTable Paths to HDFStore,closed,2010-10-07T23:42:34Z,2012-12-12T21:35:57Z,2012-12-13T01:37:53Z,4,None,...,https://github.com/pandas-dev/pandas/issues/13,"Hello,\nI not sure if this was the best way to...",2026-10-06T15:50:35.781439+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
2,358950,Enhancement,17,Better support for mixed-type data,closed,2010-10-12T16:13:55Z,2011-07-08T22:15:57Z,2011-07-08T22:15:57Z,1,None,...,https://github.com/pandas-dev/pandas/issues/17,Across DataFrame and Wide/LongPanel classes\n,2026-10-06T15:50:35.781439+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
3,358952,Bug,18,weights option may not be working in pandas.st...,closed,2010-10-12T16:15:10Z,2011-09-25T05:17:23Z,2011-12-31T16:53:33Z,1,None,...,https://github.com/pandas-dev/pandas/issues/18,Need to investigate (user notified)\n,2026-10-06T15:50:35.781439+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
4,736058,Enhancement,39,Adding variable description in DataFrame,closed,2011-04-06T00:15:26Z,2014-03-09T15:18:34Z,2014-03-09T15:18:34Z,5,None,...,https://github.com/pandas-dev/pandas/issues/39,"Hi,\n\nI think it would be great if DataFrame ...",2026-10-06T15:50:35.781439+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2584904,Enhancement|MultiIndex,499,"Clean up legacy issues in MultiIndex, have to ...",closed,2011-12-16T22:00:11Z,2016-09-29T22:09:57Z,2023-05-11T01:12:00Z,0,None,...,https://github.com/pandas-dev/pandas/issues/499,None,2026-10-06T15:50:43.270617+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
358,2585098,Bug,500,Adding DataFrames with offsets yields naive index,closed,2011-12-16T22:20:41Z,2011-12-17T18:03:57Z,2011-12-17T18:03:57Z,2,None,...,https://github.com/pandas-dev/pandas/issues/500,Adding two DataFrames with a DateRange index ...,2026-10-06T15:50:43.270617+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
359,2589627,Bug,501,Handle passing Index.name in union,closed,2011-12-17T17:38:00Z,2011-12-17T18:01:28Z,2011-12-17T18:01:28Z,0,None,...,https://github.com/pandas-dev/pandas/issues/501,,2026-10-06T15:50:43.270617+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z
360,2596810,Enhancement,502,Add R-like match function to API,closed,2011-12-19T04:26:17Z,2012-05-12T18:30:31Z,2012-05-12T18:30:31Z,1,None,...,https://github.com/pandas-dev/pandas/issues/502,,2026-10-06T15:50:43.270617+00:00,NaN,pandas-dev/pandas,Python,49923,20468,2410,2010-08-24T01:37:33Z


In [12]:
# 6. Инкрементальность
state = load_state()
df_new = incremental_filter(df, state)
state = update_watermark(df, state)
save_state(state)

In [13]:
# 7. Проверка качества
quality = check_data_quality(df_new)
print(json.dumps(quality, indent=2, ensure_ascii=False, default=str))

{
  "completeness": {
    "issue_id": 100.0,
    "labels": 100.0,
    "issue_number": 100.0,
    "issue_title": 100.0,
    "issue_state": 100.0,
    "issue_created_at": 100.0,
    "issue_closed_at": 100.0,
    "issue_updated_at": 100.0,
    "issue_comments": 100.0,
    "issue_author_association": 100.0,
    "issue_html_url": 100.0,
    "issue_body": 99.17,
    "issue__collected_at": 100.0,
    "assignees_logins": 0.55,
    "repo_full_name": 100.0,
    "repo_language": 100.0,
    "repo_stars": 100.0,
    "repo_forks": 100.0,
    "repo_open_issues": 100.0,
    "repo_created_at": 100.0
  },
  "duplicates_by_key": 0,
  "business_rule_closed_after_created": true,
  "business_rule_comments_non_negative": true
}


In [14]:
# 8. Выгрузка в csv
columns_to_save = ['repo_full_name', 'repo_created_at', 'repo_language', 'issue_number', 'issue_state', 'issue_created_at', 'issue_closed_at', 'lifetime_days', 'issue_title']
if len(df_new) > 0:
    if os.path.exists("data_clean.csv"):
        df_old = pd.read_csv("data_clean.csv")    
        pd.concat([df_old, df_new[columns_to_save]]).to_csv("data_clean.csv", index=False, encoding="utf-8")
    else:
        df_new[columns_to_save].to_csv("data_clean.csv", index=False, encoding="utf-8")
    logging.info(f"Выгружено {len(df_new)} новых строк в data_clean.csv")
else:
    logging.info(f"Нет новых строк для выгрузки в data_clean.csv")

2026-10-06 20:51:30,282 INFO Выгружено 362 новых строк в data_clean.csv
